# 04 · Validate

This notebook checks the combined table with rules written in code. Record any corrections with a reason, and save the checked table in
`data/interim/`.

In [1]:
from pathlib import Path

import pandas as pd

INTERIM = Path("../data/interim")
DOCS = Path("../docs")
DOCS.mkdir(exist_ok=True)

In [2]:
combined = pd.read_parquet(INTERIM / "combined.parquet")
checked = combined.copy()
corrections = []
checked.shape

(517, 6)

In [3]:
assert not checked.duplicated(["country_code", "year"]).any()

In [4]:
assert checked["year"].between(2014, 2024).all()
assert (checked["share_price_index"] > 0).all()

In [5]:
n_years = checked.groupby("country_code")["year"].count()
incomplete = n_years[n_years < 11]
incomplete

Series([], Name: year, dtype: int64)

In [6]:
for code in incomplete.index:
    corrections.append({
        "country_code": code,
        "rule": "all 11 years of share prices",
        "action": "country removed",
        "reason": f"only {incomplete[code]} of 11 years in the OECD data",
    })

checked = checked[~checked["country_code"].isin(incomplete.index)]
checked["country_code"].nunique()

47

In [7]:
needed = checked[checked["year"] >= 2015]
no_inflation = needed[needed["inflation_pct"].isna()]
no_inflation

,country_code,alpha2,country,year,share_price_index,inflation_pct


In [8]:
for code in no_inflation["country_code"].unique():
    years = no_inflation[no_inflation["country_code"] == code]["year"].tolist()
    corrections.append({
        "country_code": code,
        "rule": "inflation for 2015 to 2024",
        "action": "country removed",
        "reason": f"no World Bank inflation for {years}",
    })

checked = checked[~checked["country_code"].isin(no_inflation["country_code"])]
checked["country_code"].nunique()

47

In [9]:
unusual = checked[(checked["inflation_pct"] > 25) | (checked["inflation_pct"] < -2)]
unusual

,country_code,alpha2,country,year,share_price_index,inflation_pct
492,TUR,TR,Türkiye,2022,369.2984,72.308830
493,TUR,TR,Türkiye,2023,785.9030,53.859408
494,TUR,TR,Türkiye,2024,1190.7100,58.506449


In [10]:
for code in unusual["country_code"].unique():
    rows = unusual[unusual["country_code"] == code]
    corrections.append({
        "country_code": code,
        "rule": "inflation between -2% and 25%",
        "action": "kept",
        "reason": f"{len(rows)} year(s) outside the range, highest {rows['inflation_pct'].max():.1f}%. Checked against the World Bank website, the values are real.",
    })

In [11]:
checked = checked.sort_values(["country_code", "year"])
change = checked.groupby("country_code")["share_price_index"].pct_change() * 100

jumps = checked[(change > 100) | (change < -60)]
jumps

,country_code,alpha2,country,year,share_price_index,inflation_pct
493,TUR,TR,Türkiye,2023,785.903,53.859408


In [12]:
for code in jumps["country_code"].unique():
    corrections.append({
        "country_code": code,
        "rule": "yearly index change between -60% and +100%",
        "action": "kept",
        "reason": "large change in a high inflation country, the index is in local currency so prices rise with inflation",
    })

In [13]:
corrections = pd.DataFrame(corrections, columns=["country_code", "rule", "action", "reason"])
corrections.to_csv(DOCS / "corrections.csv", index=False)
corrections

,country_code,rule,action,reason
0,TUR,inflation between -2% and 25%,kept,"3 year(s) outside the range, highest 72.3%. Ch..."
1,TUR,yearly index change between -60% and +100%,kept,"large change in a high inflation country, the ..."


In [14]:
print(combined["country_code"].nunique(), "countries before,", checked["country_code"].nunique(), "after")

checked = checked.reset_index(drop=True)
checked.to_parquet(INTERIM / "checked.parquet", index=False)
checked.shape

47 countries before, 47 after


(517, 6)